# 🏃 La Grande Maratona: Chi sopravvive alla distanza?

**Data Storytelling a cura di Salvatore Rudisi**

---

## 🎯 L'Obiettivo Narrativo
Viviamo nell'era dello short content. TikTok, Reels e serie TV da binge-watching veloce. In questo contesto, chiedere a uno spettatore di impegnarsi per centinaia o migliaia di episodi sembra una follia commerciale.

In questa analisi sfideremo il dogma della "brevità" esplorando:

1. L'Illusione della Durata: Il mercato è davvero dominato dalle serie brevi?

2. Il "Muro" della Fatica: Esiste un punto di rottura matematico oltre il quale l'abbandono è inevitabile?

3. L'Antidoto della Qualità: Un punteggio elevato (Score) basta a trattenere il pubblico?

4. Tribù e Pazienza: I fan degli Shonen hanno più pazienza di quelli Slice of Life?

5. L'Anomalia dei Titani: Come fanno mostri sacri come One Piece a violare le leggi della statistica?

Utilizzeremo un dataset massiccio per rispondere alla domanda finale: **In un mondo che corre, la qualità è l'unica cosa che ci fa fermare?**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
import re
import ast
from solution.src.data import load_processed

# Settings
pd.set_option('display.max_columns', None)
warnings.filterwarnings('ignore')
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries loaded and environment configured. ✅")

Libraries loaded and environment configured. ✅


## Data Preparation & Feature Engineering 🛠️
Dopo aver caricato i dati, dobbiamo:

- **Calcolare il Drop Rate:** La nostra metrica chiave.

- **Patchare i Titani:** Inserire manualmente gli episodi per le serie in corso (es. One Piece) altrimenti verrebbero escluse dall'analisi.

- **Categorizzare la Lunghezza:** Creare fasce (Sprint vs Marathon).

In [2]:
# Load Data
df_details = load_processed("details")
df_stats = load_processed("stats")

# Merge
df = df_details.merge(df_stats, on="anime_id", how="inner")

# 1. PATCH TITANI: Inseriamo i dati reali per le big in corso (altrimenti sarebbero NaN)
titan_updates = {
    'One Piece': 1122,
    'Chibi Maruko-chan': 1512,
    'Sazae-san': 2640,
    'Nintama Rantarou': 2504,
    'Doraemon': 1787,
    'Pokemon': 1200,
    'Bleach': 366,
    'Naruto': 720
}

print("Patching Titans episode counts... 🔧")
for title, eps in titan_updates.items():
    mask = df['title'].str.contains(title, case=False, regex=False)
    df.loc[mask, 'episodes'] = df.loc[mask, 'episodes'].fillna(eps)

# 2. CLEANING: Gestione Episodi
df['episodes'] = pd.to_numeric(df['episodes'], errors='coerce')

# Invece di inventare un numero per i NaN rimasti, li rimuoviamo.
# Se non sappiamo quanto dura, non possiamo usarlo nell'analisi lunghezza vs drop.
df = df.dropna(subset=['episodes'])

# Filtro: teniamo solo serie con più di 1 episodio (via film e special)
df = df[df['episodes'] > 1]

# Calcolo Drop Rate
df['drop_rate'] = (df['dropped'] / df['members']) * 100

# Pulizia finale (rimozione NaN critici e serie con pochissimi membri)
df = df.dropna(subset=['drop_rate', 'score'])
df = df[df['members'] > 100]

# 3. Categorizzazione Lunghezza
def classify_length(ep):
    if ep < 10: return 'Short (<10)'
    elif ep <= 50: return 'Medium (10-50)'
    elif ep <= 100: return 'Long (50-100)'
    else: return 'Marathon (>100)'

df['length_class'] = df['episodes'].apply(classify_length)
length_order = ['Short (<10)', 'Medium (10-50)', 'Long (50-100)', 'Marathon (>100)']

# 4. Estrazione Anno
df['start_year'] = pd.to_datetime(df['start_date'], errors='coerce').dt.year
df['decade'] = (df['start_year'] // 10) * 10

print(f"Dataset ready: {len(df)} anime analyzed (Nessun dato inventato).")
df.head(3)

Patching Titans episode counts... 🔧
Dataset ready: 9687 anime analyzed (Nessun dato inventato).


,anime_id,title,title_japanese,url,image_url,type,status,score,scored_by,start_date,end_date,synopsis,rank,popularity,members,favorites,genres,studios,themes,demographics,source,rating,episodes,season,year,producers,explicit_genres,licensors,streaming,watching,completed,on_hold,dropped,plan_to_watch,total,score_1_votes,score_1_percentage,score_2_votes,score_2_percentage,score_3_votes,score_3_percentage,score_4_votes,score_4_percentage,score_5_votes,score_5_percentage,score_6_votes,score_6_percentage,score_7_votes,score_7_percentage,score_8_votes,score_8_percentage,score_9_votes,score_9_percentage,score_10_votes,score_10_percentage,drop_rate,length_class,start_year,decade
7,299,.hack//Liminality,.hack//LIMINALITY,https://myanimelist.net/anime/299/hack__Limina...,https://cdn.myanimelist.net/images/anime/7/230...,OVA,Finished Airing,6.58,17130,2002-06-20 00:00:00+00:00,2003-04-10 00:00:00+00:00,"While playing the newly released MMORPG ""The W...",6740,4178,34986,35,['Mystery'],['Bee Train'],['Video Game'],[],Original,PG-13 - Teens 13 or older,4,,2002,"['Bandai Visual', 'Bandai', 'CyberConnect2']",[],['Bandai Entertainment'],[],858,24478,1149,998,7504,34987,114,0.7,148,0.9,291,1.7,833,4.9,2375,13.9,3877,22.600000,4696,27.400000,2635,15.4,1229,7.2,932,5.4,2.852570,Short (<10),2002.0,2000.0
8,9332,.hack//Quantum,.hack//Quantum,https://myanimelist.net/anime/9332/hack__Quantum,https://cdn.myanimelist.net/images/anime/1894/...,OVA,Finished Airing,7.11,20412,2011-01-28 00:00:00+00:00,2011-04-07 00:00:00+00:00,"Tobias, Mary, and Sakuya challenge the impregn...",3960,3808,42217,72,"['Action', 'Adventure', 'Fantasy']",['Kinema Citrus'],['Video Game'],[],Original,PG-13 - Teens 13 or older,3,,2011,"['Bandai Visual', 'flying DOG', 'Showgate', 'B...",[],['Funimation'],['Crunchyroll'],1156,28997,635,683,10752,42223,125,0.6,65,0.3,185,0.9,522,2.6,1646,8.1,3378,16.500000,6199,30.400000,4781,23.4,2025,9.9,1487,7.3,1.617832,Short (<10),2011.0,2010.0
9,10390,.hack//Quantum: Sore ike! Bokura no Chimuchimu...,.hack//Quantum 映像特典 それいけ！ぼくらのチムチムちゃん！！,https://myanimelist.net/anime/10390/hack__Quan...,https://cdn.myanimelist.net/images/anime/4/290...,Special,Finished Airing,6.29,2810,2011-01-28 00:00:00+00:00,2011-04-07 00:00:00+00:00,DVD specials for .hack//Quantum. Main girls ar...,8559,7369,8835,3,['Comedy'],['Kinema Citrus'],['Video Game'],[],Original,PG-13 - Teens 13 or older,3,,2011,['Bandai Visual'],[],[],[],247,4643,166,293,3487,8836,66,2.3,51,1.8,68,2.4,162,5.8,391,13.9,569,20.200001,567,20.200001,431,15.3,336,12.0,169,6.0,3.316355,Short (<10),2011.0,2010.0


## Capitolo 1: L'Illusione della Durata 🌍
Prima di parlare di abbandono, dobbiamo capire la normalità. Il mercato è davvero fatto di serie brevi?

In [3]:
df_clean = df.copy()
# 1. Grafico a Torta: Distribuzione per Lunghezza
# Calcoliamo le percentuali per ogni fascia
pie_data = df['length_class'].value_counts().reset_index()
pie_data.columns = ['Lunghezza', 'Conteggio']

fig1 = px.pie(pie_data, values='Conteggio', names='Lunghezza',
             title="Circa il 60% degli anime sono sprint brevi",
             color='Lunghezza',
             # Usa una palette ordinata logicamente o distinta
             color_discrete_map={
                 'Short (<10)': '#636EFA',    # Blu (Comune)
                 'Medium (10-50)': '#00CC96', # Verde
                 'Long (50-100)': '#FFA15A',  # Arancio
                 'Marathon (>100)': '#EF553B' # Rosso (Raro)
             })

fig1.update_traces(textposition='inside', textinfo='percent+label')
fig1.show()

In [4]:
# 2 Istogramma Logaritmico (La Coda Lunga)
fig2 = px.histogram(df_clean, x='episodes', nbins=50, log_y=True,
                   title='2. Distribuzione Episodi (Scala Logaritmica)',
                   labels={'episodes': 'Numero Episodi (Bins)'},
                   color_discrete_sequence=['#EF553B'])
fig2.show()

In [5]:
# 3 Trend Temporale
trend_df = df_clean[(df_clean['start_year'] >= 1980) & (df_clean['start_year'] <= 2025)]
yearly_avg = trend_df.groupby('start_year')['episodes'].mean().reset_index()

fig3 = px.line(yearly_avg, x='start_year', y='episodes',
              title='3. Stiamo diventando più impazienti? Lunghezza Media per Anno',
              markers=True, line_shape='spline',
              labels={'episodes': 'Media Episodi', 'start_year': 'Anno'})
fig3.show()

### 📊 Insight Capitolo 1

Il panorama che emerge dai grafici è quello di un'industria in profonda mutazione:

1.  **La Dittatura dello Standard**: L'istogramma non mente. Il formato "stagionale" (12-13 o 24-26 episodi) ha ormai cannibalizzato il mercato. È un formato sicuro, digeribile e a basso rischio produttivo, diventando lo standard *de facto*.
2.  **L'Estinzione dei Dinosauri?**: Il grafico temporale (Fig 3) racconta una chiara storia di contrazione. Dagli anni '80/'90, la lunghezza media è in costante calo. L'industria si è spostata dalle "soap opera" animate infinite a prodotti chirurgici e compressi.
3.  **La Coda Lunga**: Nonostante tutto, la scala logaritmica (Fig 2) ci ricorda che le "Maratone" non sono sparite del tutto: sono diventate rare eccezioni, bestie mitologiche in un ecosistema dominato da sprint brevi e veloci.

## Capitolo 2: Il "Muro" della Fatica 🧱
Esiste una correlazione matematica tra la richiesta di tempo e la voglia di mollare?

In [6]:
# 4. Scatterplot Globale (Log Scale per leggibilità)
fig4 = px.scatter(df_clean, x='episodes', y='drop_rate',
                 hover_name='title', size='members', opacity=0.4, log_x=True,
                 title='4. Più episodi = Più Abbandoni?',
                 labels={'episodes': 'Episodi (Log Scale)', 'drop_rate': 'Drop Rate (%)'},
                 color_discrete_sequence=['#AB63FA'])
fig4.show()

In [7]:
# 5. Barplot Categorico (Il Rischio per Classe)
risk_df = df_clean.groupby('length_class')['drop_rate'].mean().reset_index()
# Ordiniamo le categorie logicamente
order = ['Sprint (1-13)', 'Medium (14-26)', 'Long (27-100)', 'Marathon (>100)']

fig5 = px.bar(risk_df, x='length_class', y='drop_rate', category_orders={'length_class': order},
             title='5. Drop Rate Medio per Tipologia',
             color='drop_rate', color_continuous_scale='Reds',
             text_auto='.1f')
fig5.show()

In [8]:
# 6. Violin Plot (La forma del rischio)
# Questo grafico mostra non solo la media, ma l'intera distribuzione del rischio
fig6 = px.violin(df, x='length_class', y='drop_rate', box=True, points=False,
                title="6. Come cambia la forma del rischio: Distribuzione Drop Rate per Lunghezza",
                labels={'length_class': 'Lunghezza', 'drop_rate': 'Drop Rate (%)'},
                category_orders={'length_class': length_order}, # Impone l'ordine logico Short -> Marathon
                color='length_class')
fig6.show()

### 📉 Insight Capitolo 2

L'analisi del Capitolo 2 conferma l'esistenza di una barriera psicologica legata alla durata:

1.  **La Fatica è Reale**: I grafici parlano chiaro, specialmente il Barplot (Fig 5). Passare da una serie "Short" a una "Marathon" comporta quasi un raddoppio del tasso di abbandono medio. Ogni episodio in più è un test per la pazienza dello spettatore.
2.  **La "Zona di Pericolo"**: La Heatmap ci mostra che il grosso degli abbandoni si concentra statisticamente quando la serie cerca di allungarsi oltre la classica singola stagione (12-24 ep), entrando in una "terra di mezzo" rischiosa.
3.  **Selezione Naturale**: Non è solo una questione di tempo, ma di aspettative. Una serie lunga chiede un investimento di vita enorme; se non ripaga immediatamente, lo spettatore "taglia le perdite" molto più velocemente rispetto a una serie breve, che spesso si finisce per inerzia anche se mediocre.

## Capitolo 3: L'Antidoto della Qualità 🛡️
Se una serie è bella, siamo disposti a perdonare la sua lunghezza?
Se piace, non si molla. L'eccellenza garantisce fedeltà.

In [9]:
# 7. Scatterplot Score vs Drop (La Discesa)
fig7 = px.scatter(df_clean, x='score', y='drop_rate', trendline="ols",
                 title='7. Correlazione Score vs Drop Rate',
                 labels={'score': 'Score Utenti', 'drop_rate': 'Drop Rate (%)'},
                 color_discrete_sequence=['#00CC96'], opacity=0.3)
fig7.show()

In [10]:
# 8. Boxplot per Fasce di Qualità
df_clean['score_bin'] = pd.cut(df_clean['score'], bins=[0, 6, 7.5, 10], labels=['Low (<6)', 'Mid (6-7.5)', 'Masterpiece (>7.5)'])

fig8 = px.box(df_clean, x='score_bin', y='drop_rate', color='score_bin',
             title='8. Drop Rate per Fascia di Voto',
             color_discrete_map={'Low (<6)': 'red', 'Mid (6-7.5)': 'orange', 'Masterpiece (>7.5)': 'green'})
fig8.show()

In [11]:
# 9. Bubble Chart Multivariato (Il Grafico Complesso)
# Filtriamo outlier estremi di members per pulizia visiva
bubble_df = df_clean[df_clean['members'] > 50000]

fig9 = px.scatter(bubble_df, x="episodes", y="drop_rate",
                 size="members", color="score",
                 log_x=True, hover_name="title",
                 title="9. Sintesi: Lunghezza (X), Abbandono (Y), Popolarità (Size), Qualità (Color)",
                 color_continuous_scale='RdYlGn') # Rosso (basso score) -> Verde (alto score)
fig9.show()

### 💎 Insight Capitolo 3

I dati del Capitolo 3 offrono una risposta chiara alla domanda "Come sopravvivere alla lunghezza?":

1.  **La Qualità come Scudo**: Esiste una correlazione inversa cristallina. Più alto è lo score, più basso è il drop rate. Lo spettatore medio è spietato con la mediocrità ma incredibilmente paziente con l'eccellenza.
2.  **La "Safe Zone" del 7.5**: I boxplot rivelano una soglia critica. Sopra il voto **7.5**, il drop rate crolla e si stabilizza verso il basso. È come se sopra quel voto gli spettatori smettessero di valutare razionalmente il "costo temporale" per godersi semplicemente il viaggio.
3.  **Il Patto con lo Spettatore (Fig 9)**: Nessuna serie riesce a diventare una "Maratona" (destra del grafico) e rimanere popolare (bolla grande) avendo una bassa qualità (colore rosso/giallo). Per chiedere centinaia di ore di vita a qualcuno, il prodotto deve essere tecnicamente o narrativamente superiore.

## Capitolo 4: Tribù e Pazienza 🎭
I fan delle commedie abbandonano più facilmente dei fan dei drama? Non tutti i generi corrono la stessa gara.

In [12]:
# --- PREPARAZIONE DATI ---
# Helper per gestire i generi (sia stringhe che liste)
def parse_genres(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except:
            return []
    return []
# Applica la funzione a tutta la colonna genres
df['genres_list'] = [parse_genres(x) for x in df['genres']]
# Esplodiamo il dataframe per avere una riga per ogni genere
df_exploded = df.explode('genres_list')
top_genres = df_exploded['genres_list'].value_counts().head(10).index.tolist()
df_genres = df_exploded[df_exploded['genres_list'].isin(top_genres)]

# 10. Barplot Lunghezza Media
# Usiamo 'genres_list' che è la colonna su cui abbiamo fatto explode
len_genre = df_genres.groupby('genres_list')['episodes'].mean().sort_values(ascending=False).reset_index()
fig10 = px.bar(len_genre, x='genres_list', y='episodes',
              title='10. Chi corre la maratona? Lunghezza media per Genere',
              color='episodes', color_continuous_scale='Blues')
fig10.show()

In [13]:
# 11. Violin Plot (Evoluzione dello Strip Plot)
# Visualizza sia la distribuzione (forma) che le statistiche (box interno)
fig11 = px.violin(df_genres, x='genres_list', y='drop_rate', color='genres_list',
                box=True,      # Aggiunge il boxplot interno per vedere la mediana
                points=False,  # Nasconde i singoli punti per un look più pulito (o usa 'outliers')
                title='11. La Pazienza delle Tribù: Densità e Rischio di Abbandono per Genere')

fig11.update_layout(
    showlegend=False, # La legenda è ridondante (già sull'asse X)
    xaxis_title="Genere",
    yaxis_title="Drop Rate"
)
fig11.show()

In [14]:
# 12. Lollipop Chart "Il Prezzo della Lunghezza" (Delta Drop Rate)
# Idea: Calcoliamo la differenza di Drop Rate tra serie Brevi (<26 ep) e Lunghe (>26 ep)

# 1. Creiamo una colonna temporanea per dividere Brevi vs Lunghe
df_genres['Length_Type'] = df_genres['episodes'].apply(lambda x: 'Breve' if x <= 26 else 'Lunga')

# 2. Calcoliamo il Drop Rate medio per le due categorie
gap_df = df_genres.groupby(['genres_list', 'Length_Type'])['drop_rate'].mean().unstack()

# 3. Calcoliamo il "Delta" (Quanto aumenta il drop rate nelle lunghe?)
# dropna() rimuove i generi che magari non hanno serie lunghe per fare il confronto
gap_df['Delta Drop'] = gap_df['Lunga'] - gap_df['Breve']
gap_df = gap_df.sort_values('Delta Drop', ascending=True).dropna()

# 4. Generiamo il grafico
fig12 = px.scatter(gap_df, x='Delta Drop', y=gap_df.index,
                   color='Delta Drop',
                   color_continuous_scale='Reds', # Più è scuro, più il genere "soffre" la lunghezza
                   title='12. Quanto aumenta l\'abbandono nelle serie lunghe?',
                   labels={'Delta Drop': 'Aumento del Drop Rate (Punti Percentuali)', 'genres_list': 'Genere'}
                  )

# Aggiungiamo le linee (steli)
for genre, row in gap_df.iterrows():
    fig12.add_shape(
        type='line',
        x0=0, y0=genre,
        x1=row['Delta Drop'], y1=genre,
        line=dict(color='gray', width=1) # Stelo grigio neutro per dar risalto al pallino
    )

# Styling
fig12.update_traces(marker=dict(size=14, opacity=1))
fig12.update_layout(xaxis_tickformat=".0%") # Formatta l'asse X come percentuale
fig12.show()

### 💡 Insight Capitolo 4
L'analisi per genere svela una verità fondamentale: **nessun genere è immune alla fatica, ma la soglia del dolore varia drasticamente.**
1.  **Non tutti corrono lo stesso rischio**: Come mostra il grafico 12, sebbene allungare una serie aumenti *sempre* il rischio di abbandono (tutti i valori sono positivi), per alcuni generi questo "prezzo da pagare" è molto più salato.
2.  **Trama vs Mood**: I generi che soffrono meno (barre più corte nel grafico 12) sono spesso quelli guidati da una trama orizzontale forte, dove lo spettatore è motivato a vedere "come va a finire".
3.  **Il pericolo della noia**: I generi spesso episodici o leggeri tendono a subire un'emorragia di pubblico maggiore quando si allungano troppo: senza un cliffhanger che incolla allo schermo, la noia vince sulla fedeltà.

## Capitolo 5: L'Anomalia dei Titani 🏴‍☠️
#### L'analisi degli Outlier.

Guardate dove sono i Titani: sono lunghissimi, ma il loro drop rate è basso come quello delle serie brevi. Sono anomalie statistiche.

In [15]:
# --- FIX ROBUSTEZZA DATI ---
# Controllo automatico: Se il drop rate medio è > 1, significa che è in scala 0-100
# Lo dividiamo per 100 per riportarlo in scala 0-1 (compatibile con i grafici %)
if df['drop_rate'].mean() > 1:
    df['drop_rate'] = df['drop_rate'] / 100
# Rimuoviamo eventuali outlier assurdi (Drop > 100%)
df = df[df['drop_rate'] <= 1.0]
# --- 1. SELEZIONE DEI "SOPRAVVISSUTI" ---
# Step A: Prendiamo i 100 anime più lunghi
top_100_long = df.sort_values('episodes', ascending=False).head(100)
# Step B: Prendiamo i 10 con il Drop Rate minore
survivors = top_100_long.sort_values('drop_rate', ascending=True).head(10)
survivors_ids = survivors['anime_id'].tolist()
long_ids = top_100_long['anime_id'].tolist()
# Etichettiamo
def tag_status(row):
    if row['anime_id'] in survivors_ids: return 'Survivor (Elite 10)'
    elif row['anime_id'] in long_ids: return 'Altri Lunghi'
    else: return 'Standard'
df['Status'] = df.apply(tag_status, axis=1)
df_long_context = df[df['anime_id'].isin(long_ids)].copy()
# --- 2. GRAFICI ---
# 13. Quadrante Magico (Corretto)
fig13 = px.scatter(df_long_context, x='score', y='drop_rate',
                  color='Status', size='members', hover_name='title',
                  title="13. Il Segreto dell'Elite: La Qualità abbatte l'Abbandono",
                  color_discrete_map={'Survivor (Elite 10)': '#00CC96', 'Altri Lunghi': 'rgba(0,0,0,0.1)'})
# Aggiungiamo linee mediane TRATTEGGIATE (dash) corrette
fig13.add_vline(x=df_long_context['score'].mean(), line_dash="dash", line_color="gray", opacity=0.3)
fig13.add_hline(y=df_long_context['drop_rate'].mean(), line_dash="dash", line_color="gray", opacity=0.3)
fig13.add_annotation(x=df_long_context['score'].max(), y=0, text="ZONA ELITE", showarrow=False, font=dict(color="#00CC96"))
fig13.update_layout(yaxis_tickformat=".0%") # Ora che i dati sono decimali (0.26), stamperà correttamente 26%
fig13.show()

In [16]:
# 14. Barplot Classifica (Ocra -> Verde)
# Scala personalizzata: Giallo Ocra (Caldo/Gold) -> Verde Smeraldo (Eccellenza)
ochre_green_scale = [
    [0.0, '#D4AC0D'],   # Giallo Ocra (Voto 7.5-8)
    [1.0, '#1E8449']    # Verde Smeraldo (Voto 9+)
]

fig14 = px.bar(survivors.sort_values('drop_rate', ascending=False), 
              x='drop_rate', y='title', orientation='h', 
              color='score', 
              title="14. I Magnifici 10: Chi sono i campioni di fedeltà?",
              labels={'drop_rate': 'Drop Rate (Più basso è meglio)', 'title': 'Anime', 'score': 'Voto'},
              color_continuous_scale=ochre_green_scale, 
              range_color=[7.8, 9.2], # Ottimizziamo il range per vedere bene la sfumatura tra i "bravi" e i "miti"
              text='drop_rate')

fig14.update_traces(texttemplate='%{text:.2%}', textposition='outside', textfont_color="black")
fig14.update_layout(xaxis_tickformat=".1%")
fig14.show()

In [17]:
# --- ASSICURIAMO CHE I DATI NECESSARI ESISTANO ---
# (Questo blocco rende la cella indipendente da esecuzioni parziali precedenti)

# Se 'survivors' non esiste, lo calcoliamo
if 'survivors' not in locals():
    # Ricreiamo la logica dei Survivors se mancante
    top_100_long = df.sort_values('episodes', ascending=False).head(100)
    survivors = top_100_long.sort_values('drop_rate', ascending=True).head(10)
    survivors_ids = survivors['anime_id'].tolist()
else:
    # Se esiste survivors, dobbiamo comunque assicurarci di avere survivors_ids
    if 'survivors_ids' not in locals():
        survivors_ids = survivors['anime_id'].tolist()
    if 'top_100_long' not in locals():
        top_100_long = df.sort_values('episodes', ascending=False).head(100)

# DEFINIAMO OTHERS_LONG (La causa dell'errore)
others_long = top_100_long[~top_100_long['anime_id'].isin(survivors_ids)]


# --- 15. Dumbbell Plot: Il Divario Orizzontale (Leggibile) ---
# Ora possiamo procedere col grafico sicuri che le variabili esistano

# Calcoliamo le medie
metrics = ['Qualità (Score)', 'Popolarità (Membri)', 'Lunghezza (Episodi)']
elite_vals = [survivors['score'].mean(), survivors['members'].mean(), survivors['episodes'].mean()]
others_vals = [others_long['score'].mean(), others_long['members'].mean(), others_long['episodes'].mean()]

# Creiamo un DataFrame per il plot
data = []
# Logica di formattazione del testo per rendere i numeri belli da leggere
def fmt(val, metric):
    if 'Membri' in metric: return f"{val/1_000_000:.1f}M" # Es: 1.5M
    if 'Score' in metric: return f"{val:.2f}"           # Es: 8.54
    return f"{val:.0f}"                                 # Es: 145

for m, e, o in zip(metrics, elite_vals, others_vals):
    # Normalizziamo su scala 0-1 per disegnare le linee (altrimenti i milioni schiacciano i voti)
    max_val = max(e, o) * 1.2 # 20% di margine per estetica
    
    data.append({
        'Metrica': m, 'Gruppo': 'Elite Survivors', 'Valore_Norm': e/max_val, 'Valore_Reale': e,
        'Label': fmt(e, m), 'Color': '#00CC96'
    })
    data.append({
        'Metrica': m, 'Gruppo': 'Altri Lunghi', 'Valore_Norm': o/max_val, 'Valore_Reale': o,
        'Label': fmt(o, m), 'Color': 'gray'
    })

df_dumb = pd.DataFrame(data)

fig15 = go.Figure()

# 1. Disegniamo le linee grigie (i manubri)
for m in metrics:
    row_elite = df_dumb[(df_dumb['Metrica'] == m) & (df_dumb['Gruppo'] == 'Elite Survivors')]
    row_other = df_dumb[(df_dumb['Metrica'] == m) & (df_dumb['Gruppo'] == 'Altri Lunghi')]
    
    fig15.add_shape(type="line",
                    y0=m, x0=row_other['Valore_Norm'].values[0],
                    y1=m, x1=row_elite['Valore_Norm'].values[0],
                    line=dict(color="lightgray", width=4))

# 2. Aggiungiamo i punti sopra
fig15.add_trace(go.Scatter(
    y=df_dumb['Metrica'],
    x=df_dumb['Valore_Norm'],
    mode='markers+text',
    marker=dict(color=df_dumb['Color'], size=18, line=dict(width=2, color='white')),
    text=df_dumb['Label'],
    textposition="top center",
    textfont=dict(size=12, color='black'),
    showlegend=False
))

# Legenda Manuale (Trucco per farla apparire corretta nel Dumbbell Plot)
fig15.add_trace(go.Scatter(x=[None], y=[None], mode='markers', marker=dict(color='#00CC96', size=15), name='Elite Survivors'))
fig15.add_trace(go.Scatter(x=[None], y=[None], mode='markers', marker=dict(color='gray', size=15), name='Altri Lunghi'))

fig15.update_layout(
    title="15. Il Divario: Elite vs Altri (Confronto Diretto)",
    xaxis=dict(showgrid=False, showticklabels=False, range=[-0.1, 1.1]), # Nascondiamo asse X (usiamo le etichette)
    yaxis=dict(showgrid=True, gridcolor='rgba(0,0,0,0.1)'),
    margin=dict(l=20, r=20, t=60, b=20),
    legend=dict(orientation="h", yanchor="bottom", y=1, xanchor="right", x=1)
)

fig15.show()

### Insight Capitolo 5

Analizzando l'élite dei 10 anime più fedeli tra i lunghissimi (Fig 15), emerge chiaramente cosa fa la differenza:

1.  **Qualità Superiore**: Il divario più netto è nello **Score**. I "Survivors" hanno una media voto nettamente superiore agli altri lunghi. Per tenere incollato lo spettatore per 100+ episodi, non basta la trama, serve l'eccellenza.
2.  **Meno è Meglio (Relativamente)**: È interessante notare (Fig 15, pannello 3) che i "Survivors" hanno mediamente *meno* episodi rispetto al resto della Top 100. Sembra esserci un limite umano anche per l'eccellenza: è più facile mantenere la qualità (e il pubblico) per 150 episodi che per 800.
3.  **Il Caso "Gintama"**: Guardando i titoli della Fig 14, si nota spesso la dominanza di franchise specifici (come *Gintama* o *Legend of Galactic Heroes*) che hanno costruito fanbase di nicchia ma estremamente devote, al contrario dei "blockbuster" generalisti.

### Conclusioni Finali🔚

In [18]:
# --- GRAFICO FINALE RIASSUNTIVO ---

# 0. SICUREZZA: Ricreiamo la colonna score_class se manca (per evitare KeyError)
def classify_score_safe(s):
    if s < 6: return 'Low (<6)'
    elif s <= 7.5: return 'Mid (6-7.5)'
    else: return 'High (>7.5)'

# La applichiamo sempre per sicurezza
df['score_class'] = df['score'].apply(classify_score_safe)

# 16. La Curva della Sopravvivenza (Lineplot con Smoothing)
# Riassume tutto: Come cambia il drop rate all'aumentare degli episodi?
# E come la qualità (Score) cambia la pendenza della curva?

# Creiamo una copia per lavorare puliti
df_final = df.copy()

# Creiamo fasce di episodi arrotondate per ridurre il rumore e creare una linea pulita
df_final['episodes_bin'] = df_final['episodes'].round(-1) # Arrotonda ai 10 episodi più vicini
df_final = df_final[df_final['episodes_bin'] <= 200] # Ci concentriamo sulla massa critica (<200 ep) per chiarezza

# Calcoliamo la media per bin e per fascia di voto
survival_curve = df_final.groupby(['episodes_bin', 'score_class'])['drop_rate'].mean().reset_index()

# Ordiniamo le classi di voto per il grafico
category_order = ['Low (<6)', 'Mid (6-7.5)', 'High (>7.5)']

fig16 = px.line(survival_curve, x='episodes_bin', y='drop_rate', color='score_class',
               title='16. La Sintesi: Tre strade per il destino',
               labels={'episodes_bin': 'Numero di Episodi (Durata)', 'drop_rate': 'Tasso di Abbandono Medio'},
               category_orders={'score_class': category_order},
               color_discrete_map={
                   'Low (<6)': '#EF553B',   # Rosso (Strada della Morte)
                   'Mid (6-7.5)': '#FFA15A',# Arancio (Strada Faticosa)
                   'High (>7.5)': '#00CC96' # Verde (Autostrada del Successo)
               },
               markers=True)

# Styling per renderlo un "Grand Finale"
fig16.update_layout(
    xaxis_title="Lunghezza della Maratona (Episodi)",
    yaxis_tickformat=".0%",
    legend_title="Qualità (Score)",
    hovermode="x unified"
)
fig16.show()

Abbiamo iniziato chiedendoci se la lunghezza fosse una condanna a morte per un anime. I dati hanno risposto con una sfumatura cruciale: **la lunghezza è un amplificatore.**

### I 3 Pilastri della Sopravvivenza:

1.  🔴 **La Condanna della Mediocrità**: Per le serie di bassa qualità (*Score < 6*), ogni episodio aggiuntivo è un chiodo nella bara. La curva rossa nel grafico finale (Fig 16) mostra un'impennata verticale del drop rate: lo spettatore non ha pazienza per il tempo sprecato.
2.  🟠 **La Fatica della Norma**: Per la maggior parte degli anime (*Mid 6-7.5*), la lunghezza è un rischio calcolato. Più si allunga la storia, più si perde pubblico fisiologicamente. È una guerra di logoramento.
3.  🟢 **L'Immunità dell'Eccellenza**: La scoperta più affascinante è la curva verde. Per i capolavori (*Score > 7.5*), **la lunghezza diventa irrilevante**. La curva è piatta. Quando la qualità è altissima, lo spettatore smette di contare i minuti e inizia a desiderare che il viaggio non finisca mai.

**In sintesi**: Non è la lunghezza a uccidere l'interesse, è l'incapacità di giustificarla. *One Piece* e i *Survivors* non sono eccezioni statistiche, sono la prova che se il contenuto vale, il tempo dello spettatore è infinito.